# 05_test — Test 평가 6-A·6-B·6-C, 견고성, 민감도 (초안: core 담당 2026-11-26, 실행·검증: eval 담당)

**사전 고정(`99_freeze`) 뒤 한 번만** 실행합니다(`DRY_RUN = False` 상태로 저장·고정한 뒤, 승인 후 노트북 파일을 고쳐 저장하지 않고 그대로 실행).

- **코드 점검(`LC_TEST_DRYRUN=1`)**: Test를 열지 않습니다. 노트북 파일을 고쳐 저장하지 않고 환경변수 `LC_TEST_DRYRUN=1`을 주면 `DRY_RUN=True`로 동작해 Dev의 부트스트랩 b=0 OOB를 가짜 Test로, `models_b/model_b000`을 최종 모형으로 씁니다. 결과는 `report/_dryrun/`에만 저장하며, `selection.json`은 입력 지문이 현재 산출물과 모두 일치할 때만 재사용합니다.
- **실제 본 실행(`DRY_RUN = False`, 기본값)**: Test를 열기 **전에** `load_frozen_test_inputs()`가 사전 고정본(`05_test.ipynb` 코드 해시 포함)·`selection.json`·Test parquet 해시를 검증하고, 고정된 `selection.json`에서 **선택 모델과 모형별 a\*(06_collect의 SR 중앙값 곡선 최대)**를 받습니다. 그 뒤 `test_hold/test_X_raw.parquet`, `test_meta.parquet`(00b_test_prep 결과)와 `outputs/*/final_model.joblib`을 씁니다.
- a\*: 주 분석(H1, 표 6)은 ②의 a\*를 세 모형 공통으로, 표 A는 모형마다 자기 a\*를 씁니다. DRY_RUN에서 유효·일치하는 `selection.json`이 없으면 `bootstrap_curves.csv`로 같은 규칙(중앙값 곡선 최대)을 직접 계산합니다.

미정 항목은 그대로 돌고 결과에 `[미정]`으로 남습니다: δ(`DELTA`), B_TEST, PoM 정의, 경계 사례 범위, Jaccard 대상 차주 수.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
import os
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
%run -i ./lib_eval.ipynb
# 사전 고정(99_freeze) 대상 파일은 DRY_RUN=False 상태로 저장·고정하고, 승인 후 코드를 수정하지 않고 본 실행합니다.
# 사전 고정 전 시험 실행은 이 파일을 고치지 말고 환경변수 LC_TEST_DRYRUN=1 로 실행하세요.
DRY_RUN = os.environ.get("LC_TEST_DRYRUN", "0").strip().lower() in ("1", "true", "yes")
if not DRY_RUN:
    assert STAGE == "final" and not PENDING, f"사전 고정 뒤에만 실행합니다. 미결: {list(PENDING)} (시험 실행은 LC_TEST_DRYRUN=1)"
set_korean_font()
TAB = REPORT_DIR / ("_dryrun" if DRY_RUN else "tables"); FIG = REPORT_DIR / ("_dryrun" if DRY_RUN else "figures")
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
N_JACCARD_OBS = 500     # [미정·기본값] 안정성을 볼 차주 수: 세 모형이 모두 거절한 차주에서 무작위 500명
BOUNDARY_BAND = 0.01    # [미정·기본값] '커트라인 바로 밖' = ②의 승인 경계 다음 1%(승인율 기준) 안의 거절 차주

if not DRY_RUN:   # Test를 열기 전에 사전 고정 검증 (실패하면 여기서 멈춤)
    GATE = load_frozen_test_inputs()

X_dev = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet"); meta_dev = pd.read_parquet(DATA_DIR / "dev_meta.parquet")
if DRY_RUN:
    tr0 = np.random.default_rng(SEED).integers(0, len(X_dev), size=len(X_dev), dtype=np.int64)   # make_oob_splits 첫 회차
    oob0 = np.setdiff1d(np.arange(len(X_dev)), tr0)
    X_t, meta_t = X_dev.iloc[oob0].reset_index(drop=True), meta_dev.iloc[oob0].reset_index(drop=True)
    FILES = {m: OUT_DIR / m / "models_b" / "model_b000.joblib" for m in MODELS}
else:
    X_t = pd.read_parquet(ROOT / "test_hold" / "test_X_raw.parquet"); meta_t = pd.read_parquet(ROOT / "test_hold" / "test_meta.parquet")
    FILES = {m: OUT_DIR / m / "final_model.joblib" for m in MODELS}
SUM = {m: json.load(open(OUT_DIR / m / "summary.json")) for m in MODELS}
for m, s in SUM.items():   # 06_collect와 같은 버전 점검
    assert s["config_hash"] == config_fingerprint() and s["lib_prep_hash"] == LIB_PREP_HASH, f"{m} 결과가 지금 버전이 아님"

def median_curve_astar(m):
    """06_collect 2-1과 같은 규칙: 승인율별 OOB SR B개의 중앙값 곡선에서 MIN_RATE 이상 최댓값 (유효 SR이 B개인 승인율만)."""
    cur = pd.read_csv(OUT_DIR / m / "bootstrap_curves.csv").assign(rate=lambda d: d["rate"].round(4))
    g = cur[cur["rate"] >= MIN_RATE - 1e-9].groupby("rate")["sr"]
    med = g.median()[g.count() == cur["b"].nunique()].dropna()
    return float(med.idxmax())

def _selection_is_fresh(sel):
    """DRY_RUN에서도 selection.json이 현재 산출물·설정에서 나왔는지 입력 파일 SHA-256까지 확인한다."""
    if sel.get("valid") is not True:
        return False, f"valid={sel.get('valid')}"
    if sel.get("lib_prep_hash") != LIB_PREP_HASH:
        return False, "lib_prep_hash 불일치"
    if sel.get("config_hash") != config_fingerprint():
        return False, "config_hash 불일치"
    if sel.get("ref_model", REF_MODEL) != REF_MODEL:
        return False, "ref_model 불일치"
    a_map = sel.get("a_star")
    if not isinstance(a_map, dict) or not all(m in a_map for m in MODELS):
        return False, "a_star 모형 누락"
    sel_inputs = sel.get("inputs") or {}
    req_inputs = {f"{m}/{fn}" for m in MODELS for fn in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib")}
    if not req_inputs.issubset(sel_inputs.keys()):
        return False, "inputs 필수 항목 누락"
    for rel, expected_sha in sel_inputs.items():
        fp = OUT_DIR / rel
        if not fp.exists() or sha256_file(fp) != expected_sha:
            return False, f"입력 파일 변경/누락({rel})"
    return True, "OK"

if DRY_RUN:
    _sel_path = OUT_DIR / "collect" / "selection.json"
    _sel = json.loads(_sel_path.read_text(encoding="utf-8")) if _sel_path.exists() else {}
    _sel_ok, _sel_reason = _selection_is_fresh(_sel) if _sel else (False, "selection.json 없음")
    if _sel_ok:
        A_STAR_BY, SELECTED = _sel["a_star"], _sel["selected_model"]
    else:
        A_STAR_BY, SELECTED = {m: median_curve_astar(m) for m in MODELS}, None
        print(f"DRY_RUN: 최신 selection.json 미사용({_sel_reason}) → bootstrap_curves.csv로 중앙값 곡선 a*를 직접 계산")
else:
    A_STAR_BY, SELECTED = GATE["a_star_by_model"], GATE["selected_model"]   # 고정된 selection.json (06_collect)
A_STAR_BY = {m: float(A_STAR_BY[m]) for m in MODELS}
A_STAR = A_STAR_BY[REF_MODEL]                        # 주 분석: ②의 a*를 세 모형 공통으로 (H1)
y_t, e_t = meta_t["y"].to_numpy(), meta_t["excess_ret"].to_numpy()
FITTED = {m: joblib.load(f) for m, f in FILES.items()}
P = {m: predict(*FITTED[m], X_t) for m in MODELS}
print(f"{'DRY_RUN (Dev b=0 OOB)' if DRY_RUN else 'Test'} {len(X_t):,}건, 부도율 {y_t.mean():.4f} | 공통 a* = {A_STAR} | 모형별 a* = {A_STAR_BY} | 선택 = {SELECTED} | δ = {DELTA} | B_TEST = {B_TEST}")

## 1. 6-A 판별력·보정 → 표 5

In [ ]:
t5 = metrics_6a(y_t, P)
t5.insert(1, "모형", t5["model"].map(MODELS))
# [미정·권고] E-6: Dev OOB AUC(부트스트랩 평균)와의 차이 = Dev y 사용에 따른 낙관의 크기
t5["oob_auc_mean"] = [pd.read_csv(OUT_DIR / m / "bootstrap_results.csv")["auc"].mean() for m in t5["model"]]
t5["auc_gap_test_minus_oob"] = t5["auc"] - t5["oob_auc_mean"]
t5.round(4).to_csv(TAB / "t5_test_6a.csv", index=False, encoding="utf-8-sig")
t5.round(4)

## 2. 6-B SR·ΔSR, 비열등 판정 → 표 6, 그림 5·6

In [ ]:
SR = {m: sr_point(P[m], e_t, A_STAR) for m in MODELS}
t0 = time.time()
BS = paired_bootstrap_sr(P, e_t, A_STAR, B_TEST, seed=SEED)
sec = time.time() - t0
print(f"paired bootstrap {B_TEST}회: {sec:.0f}초  (B_TEST 규칙: 4,999회가 10분 안이면 4,999 [미정])")
rows = [noninferiority_row("② − ③", SR["xgb"] - SR["xgb_mono"], BS["xgb"] - BS["xgb_mono"], DELTA, sr_ref=SR["xgb"]),
        noninferiority_row("② − ①", SR["xgb"] - SR["logit"], BS["xgb"] - BS["logit"], None, sr_ref=SR["xgb"])]
rows[1]["판정"] = "보고만 (가설 아님)"
t6 = pd.DataFrame(rows)
for m in MODELS: t6[f"SR_{m}"] = SR[m]
t6["B_TEST"] = B_TEST; t6["a_star"] = A_STAR
t6.round(5).to_csv(TAB / "t6_test_6b.csv", index=False, encoding="utf-8-sig")
# [미정] E-5 결과 해석표의 '≈' 정의는 δ와 함께 정함. 여기서는 숫자만 냄
t6.round(5)

In [ ]:
# 표 A: 전략별 성과 (설계 '표 A'). 각 모형은 자기 a*, '전부 승인'은 승인율 1.0. OOB SR은 자기 a*에서 회차별 OOB SR의 중앙값
ROLE = {"xgb": "기준: 제약 없는 최고 성능", "logit": "기준: 전통 방법", "xgb_mono": "후보: 단조 제약"}
def oob_sr_at(m, a):
    cur = pd.read_csv(OUT_DIR / m / "bootstrap_curves.csv")
    return float(cur.loc[np.isclose(cur["rate"], a), "sr"].median())
tA = pd.DataFrame([dict(전략=MODELS[m], model=m, 역할=("최종 선택" if m == SELECTED else ROLE[m]), 승인율=A_STAR_BY[m],
                        OOB_SR=oob_sr_at(m, A_STAR_BY[m]), Test_SR=sr_point(P[m], e_t, A_STAR_BY[m])) for m in MODELS]
                  + [dict(전략="전부 승인", model="approve_all", 역할="기준: 모델 없음", 승인율=1.0,
                          OOB_SR=sr_point(np.zeros(len(meta_dev)), meta_dev["excess_ret"].to_numpy(), 1.0),   # 모형이 없어 OOB 대신 Dev 전체
                          Test_SR=sr_point(np.zeros(len(e_t)), e_t, 1.0))])
if SELECTED is None:
    print("선택 전(selection.json 없음): '최종 선택' 표시 없음")
tA.round(5).to_csv(TAB / "tA_strategy_sr.csv", index=False, encoding="utf-8-sig")
tA.round(5)


In [ ]:
# B_TEST 결정용(eval 담당): seed만 바꿔 97.5% 분위수가 얼마나 흔들리나
if "B_TEST" in PENDING:
    q = [np.quantile((lambda b: b["xgb"] - b["xgb_mono"])(paired_bootstrap_sr(P, e_t, A_STAR, B_TEST, seed=s)), 0.975) for s in (SEED + 1, SEED + 2)]
    print(f"seed 42/43/44의 97.5% 분위수: {t6.loc[0, 'q_975']:.5f}, {q[0]:.5f}, {q[1]:.5f}")

In [ ]:
# 그림 5: ②−③ paired bootstrap 분포
d = BS["xgb"] - BS["xgb_mono"]
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(d, bins=60, color="C2", alpha=0.7)
ax.axvline(t6.loc[0, "q_975"], color="k", ls="--", lw=1, label=f"97.5% 분위수 {t6.loc[0, 'q_975']:.4f}")
if DELTA is not None:
    ax.axvline(DELTA, color="C3", lw=1.5, label=f"δ = {DELTA}")
else:
    ax.text(0.98, 0.9, "δ 미정", transform=ax.transAxes, ha="right", color="C3")
ax.set_xlabel("ΔSR = SR② − SR③"); ax.set_ylabel("회차 수"); ax.legend(frameon=False, fontsize=8); tidy(ax)
fig.tight_layout(); fig.savefig(FIG / "fig5_test_dsr_boot.png", dpi=300); plt.show()

# 그림 6: 승인율별 ΔSR (점추정), 견고성 구간 음영
rates = RATES[(RATES >= 0.10 - 1e-9) & (RATES <= 0.90 + 1e-9)]
curve = sr_by_rate(P, e_t, rates)
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.axvspan(*ROBUST["rate_band"], color="0.92")
ax.plot(rates, curve["xgb"] - curve["xgb_mono"], color="C2", label="② − ③")
ax.plot(rates, curve["xgb"] - curve["logit"], color="C0", label="② − ①")
ax.axhline(0, color="k", lw=0.8); ax.axvline(A_STAR, color="0.4", ls=":", lw=1)
ax.set_xlabel("승인율 a"); ax.set_ylabel("ΔSR (Test)"); ax.legend(frameon=False, fontsize=8); tidy(ax)
fig.tight_layout(); fig.savefig(FIG / "fig6_test_dsr_band.png", dpi=300); plt.show()
curve.to_csv(TAB / "t6b_sr_by_rate.csv", encoding="utf-8-sig")

## 3. 견고성 → 표 7

In [ ]:
t7 = robustness_table(P, meta_t, A_STAR)
t7.round(5).to_csv(TAB / "t7_robustness.csv", index=False, encoding="utf-8-sig")
t7.round(5)

## 4. 6-C 단조성 위반율, Top-3 거절 사유 안정성 → 표 8, 그림 7·8

In [ ]:
SD = X_dev[list(SIGN_OF)].std()             # 원 단위 Dev 표준편차
rows = []
for m in MODELS:
    rej = ~approve_mask(P[m], A_STAR)          # 그 모형이 거절한 차주
    v = violations(*FITTED[m], X_t.loc[rej], SD)
    rows.append(dict(model=m, 모형=MODELS[m], n_rejected=int(rej.sum()), viol_borrower=v["borrower_rate"], viol_pair=v["pair_rate"]))
    if m == "xgb_mono" and v["borrower_rate"] > 0:
        print("경고: ③에서 위반이 나옴 → 부호표·전처리 확인", v["by_var"][v["by_var"] > 0])
t8 = pd.DataFrame(rows)

rej_all = np.flatnonzero(np.logical_and.reduce([~approve_mask(P[m], A_STAR) for m in MODELS]))
obs = np.random.default_rng(SEED).choice(rej_all, min(N_JACCARD_OBS, len(rej_all)), replace=False)
ref_X = X_dev.sample(50_000, random_state=SEED)   # 로지스틱 기여도의 기준점 x̄ [미정·기본값: Dev 평균]
t0 = time.time()
for m in MODELS:
    files = sorted((OUT_DIR / m / "models_b").glob("model_b*.joblib"))
    st = stability(files, X_t.iloc[obs], ref_X=ref_X)
    for k in ("n_models", "n_pairs", "jaccard", "jaccard_sd", "rbo"):
        t8.loc[t8.model == m, k] = st[k]
print(f"안정성 {time.time() - t0:.0f}초")
t8.round(4).to_csv(TAB / "t8_test_6c.csv", index=False, encoding="utf-8-sig")
t8.round(4)

In [ ]:
# 그림 7: 위반율과 Jaccard
fig, ax = plt.subplots(1, 2, figsize=(6.5, 2.8))
lab = [MODELS[m] for m in t8.model]
ax[0].bar(lab, t8.viol_borrower * 100, color=["C0", "C1", "C2"]); ax[0].set_ylabel("위반 차주 비율 (%)")
ax[1].bar(lab, t8.jaccard, color=["C0", "C1", "C2"]); ax[1].set_ylabel("Top-3 Jaccard (평균)"); ax[1].set_ylim(0, 1)
for a_, vals, fmt in ((ax[0], t8.viol_borrower * 100, "{:.1f}%"), (ax[1], t8.jaccard, "{:.2f}")):
    for x_, v_ in zip(lab, vals): a_.text(x_, v_, fmt.format(v_), ha="center", va="bottom", fontsize=7)
for a_ in ax: a_.tick_params(axis="x", labelsize=7); tidy(a_)
fig.tight_layout(); fig.savefig(FIG / "fig7_violation_jaccard.png", dpi=300); plt.show()

# 그림 8: 경계 사례 3명 (AUDIT_CASES: 무작위, seed 고정, 결과 보기 전 규칙 고정)
order = np.argsort(P[REF_MODEL], kind="stable"); k = n_approve(len(order), A_STAR)
band = order[k: k + max(int(BOUNDARY_BAND * len(order)), AUDIT_CASES["n"])]
cases = np.random.default_rng(AUDIT_CASES["seed"]).choice(band, AUDIT_CASES["n"], replace=False)
fig, ax = plt.subplots(AUDIT_CASES["n"], 3, figsize=(6.5, 2.2 * AUDIT_CASES["n"]), squeeze=False)
case_rows = []
for i, c in enumerate(cases):
    for j, m in enumerate(MODELS):
        prep, model = FITTED[m]
        ref_mean = prep.transform(ref_X).mean() if m == "logit" else None
        C = contributions(prep, model, X_t.iloc[[c]], ref_mean).iloc[0].sort_values(ascending=False).head(5)[::-1]
        ax[i, j].barh(C.index, C.values, color=["C3" if v > 0 else "C0" for v in C.values])
        ax[i, j].tick_params(labelsize=6); tidy(ax[i, j])
        if i == 0: ax[i, j].set_title(MODELS[m], fontsize=8)
        case_rows.append(dict(case=i + 1, row=int(c), model=m, p_hat=float(P[m][c]),
                              top3=", ".join(top_reasons(contributions(prep, model, X_t.iloc[[c]], ref_mean))[0])))
    ax[i, 0].set_ylabel(f"사례 {i + 1}", fontsize=8)
fig.supxlabel("로그오즈 기여 (위 5개)", fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "fig8_boundary_cases.png", dpi=300); plt.show()
pd.DataFrame(case_rows).to_csv(TAB / "t8b_boundary_cases.csv", index=False, encoding="utf-8-sig")

## 5. 민감도 A: Core 16 → 표 9 (07_core16 결과가 있을 때만)

In [ ]:
pre = "_dryrun_" if DRY_RUN else ""
core_files = {m: OUT_DIR / "core16" / f"{pre}{m}.joblib" for m in MODELS}
if all(f.exists() for f in core_files.values()):
    core_cols = json.load(open(OUT_DIR / "core16" / f"{pre}summary.json"))["core_cols"]
    PC = {m: predict(*joblib.load(f), X_t[core_cols]) for m, f in core_files.items()}
    SRC = {m: sr_point(PC[m], e_t, A_STAR) for m in MODELS}
    t9 = pd.DataFrame([dict(분석=n, SR_logit=s["logit"], SR_xgb=s["xgb"], SR_xgb_mono=s["xgb_mono"],
                            dSR_2_3=s["xgb"] - s["xgb_mono"], dSR_2_1=s["xgb"] - s["logit"],
                            AUC_xgb=roc_auc_score(y_t, p["xgb"]))
                       for n, s, p in (("주 분석 37개", SR, P), ("민감도 A Core 16", SRC, PC))])
    t9["부호 같음(②−③)"] = np.sign(t9["dSR_2_3"]).eq(np.sign(t9.loc[0, "dSR_2_3"]))
    t9.round(5).to_csv(TAB / "t9_sensitivity_core16.csv", index=False, encoding="utf-8-sig")
    display(t9.round(5))
else:
    print("07_core16 결과 없음 → 표 9 건너뜀:", [f.name for f in core_files.values() if not f.exists()])

In [ ]:
# 6. 기록
rec = dict(dry_run=DRY_RUN, n=len(X_t), a_star=A_STAR, a_star_by_model=A_STAR_BY, selected_model=SELECTED, delta=DELTA, B_TEST=B_TEST, bootstrap_seconds=round(sec),
           pending=sorted(PENDING), config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH,
           undecided_in_code=dict(N_JACCARD_OBS=N_JACCARD_OBS, BOUNDARY_BAND=BOUNDARY_BAND), meta=run_meta())
(TAB / "test_summary.json").write_text(json.dumps(rec, ensure_ascii=False, indent=1, default=str))
print("저장:", sorted(p.name for p in TAB.glob("*")))